In [ ]:
import jams
import numpy as np
import os
import pretty_midi
import random
np.int = int # deprecated np.int
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
from collections import defaultdict
import math

In [4]:
midi_dir= '/data/akshaj/MusicAI/GuitarSet/MIDIAnnotations/'

In [19]:
def encode_notes_for_test(jam, technique_probability=0.5):
    '''
    Adds sample random expressive techniques, random strings, and frets 
    for testing/tab output purposes
    
    Args:
        jam: Input JAMS object
        technique_probability: Probability of adding a technique (0.0-1.0)
    '''
    new_ann = jams.Annotation(namespace='tab_note')
    
    # Valid technique options (removed None!)
    tech_options = ["slide", "vibrato", "hammer-on", "pull-off", "bend", "harmonic"]
    
    technique_count = 0
    
    # Iterate over existing notes
    note_ann = jam.annotations[0]  
    for obs in note_ann.data:
        pitch = obs.value
        
        # Create techniques list (may be empty)
        techniques = []
        if random.random() < technique_probability:
            techniques = [random.choice(tech_options)]
            technique_count += 1

        value = {
            "pitch": pitch,
            "string": random.randint(1, 6),
            "fret": random.randint(0, 12),
            "techniques": techniques  # Clean list (no None values)
        }

        new_ann.append(
            time=obs.time, 
            duration=obs.duration, 
            value=value, 
            confidence=obs.confidence
        )

    # Add new tab_note annotation
    jam.annotations.append(new_ann)
    
    print(f"\n✓ Created {len(new_ann.data)} notes")
    print(f"✓ Added {technique_count} techniques ({technique_count/len(new_ann.data)*100:.1f}%)")
    
    return jam

In [27]:
def add_random_techniques_to_existing_jam(jam, technique_probability=0.5):
    '''
    Adds random techniques to EXISTING tab_note annotation
    '''
    tech_options = ["slide", "vibrato", "hammer-on", "pull-off", "bend", "harmonic"]
    
    # Find the EXISTING tab_note annotation
    tab_ann = None
    for ann in jam.annotations:
        if ann.namespace == 'tab_note':
            tab_ann = ann
            break
    
    if tab_ann is None:
        raise ValueError("No tab_note annotation found! Run midi_to_jams_with_tablature() first")
    
    technique_count = 0
    
    # MODIFY existing notes (don't create new ones)
    for obs in tab_ann.data:
        # Add technique to existing note
        if random.random() < technique_probability:
            tech = random.choice(tech_options)
            obs.value['techniques'] = [tech]  # ← Modify existing
            technique_count += 1
        else:
            obs.value['techniques'] = []
    
    print(f"\n✓ Modified {len(tab_ann.data)} notes")
    print(f"✓ Added {technique_count} techniques ({technique_count/len(tab_ann.data)*100:.1f}%)")
    
    return jam

In [ ]:
from real_tab_converter import midi_to_jams_with_tablature, jams_to_tablature_only

# Step 1: Create JAMS with tablature
jam = midi_to_jams_with_tablature('song.mid')

# Step 2: Add techniques to EXISTING tab_note annotation
jam = add_random_techniques_to_existing_jam(jam, technique_probability=0.5)

# Step 3: Debug - check techniques are there
tab_ann = [a for a in jam.annotations if a.namespace == 'tab_note'][0]
print("\nNotes with techniques:")
for i, obs in enumerate(tab_ann.data[:20]):
    techs = obs.value.get('techniques', [])
    if techs:
        print(f"  Note {i}: {techs}")

# Step 4: Convert to MusicXML
jams_to_tablature_only(jam, 'techniques.xml', tempo_bpm=108)

ImportError: cannot import name 'jams_to_tablature_only' from 'real_tab_converter' (/data/colin-obrien/Music-AI/4_Tabs/real_tab_converter.py)

In [26]:
# Test to see if we can get epxressive techniques into the tablature
from real_tab_converter import midi_to_jams_with_tablature, jams_to_musicxml_real
import pretty_midi

# Test midi file
midi_path = os.path.join(midi_dir, '05_Funk2-108-Eb_solo.mid')

# Startup code
pm = pretty_midi.PrettyMIDI(midi_path)
# Try to get tempo from MIDI
tempo_changes = pm.get_tempo_changes()
if len(tempo_changes[1]) > 0:
    tempo_bpm = int(tempo_changes[1][0])
    print(f"\nDetected tempo: {tempo_bpm} BPM")
else:
    tempo_bpm = 120
    print(f"\nUsing default tempo: {tempo_bpm} BPM")

# Create jams object out of midi data
jam = midi_to_jams_with_tablature(midi_path)

# Populate jams object with random expressive techniques
jam = encode_notes_for_test(jam)

# Debug: Check what was actually created
tab_ann = [a for a in jam.annotations if a.namespace == 'tab_note'][1]
print("\nFirst 10 notes:")
for i, obs in enumerate(tab_ann.data[:10]):
    techs = obs.value.get('techniques', [])
    print(f"  Note {i}: {techs}")

# Convert to MusicXML and ouptut file
output_file = jams_to_musicxml_real(jam, 'techniques.xml', tempo_bpm)


Detected tempo: 60 BPM
Converted 61 notes to tablature
Pitch range: 56 - 75

✓ Created 61 notes
✓ Added 34 techniques (55.7%)

First 10 notes:
  Note 0: []
  Note 1: []
  Note 2: ['harmonic']
  Note 3: ['hammer-on']
  Note 4: ['slide']
  Note 5: ['bend']
  Note 6: ['pull-off']
  Note 7: ['harmonic']
  Note 8: ['slide']
  Note 9: ['vibrato']
Converting 61 notes to MusicXML...
Test
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques found
No techniques fo